# 01 — Data Wrangling

Loads raw CSVs, builds treatment/outcome flags, performs feature engineering, and writes a single clean analysis table consumed by all downstream notebooks.

Output: `notebooks/processed/df_analysis.csv`
Downstream modeling notebooks should use `helpers.py` for consistent post-load prep.

## Setup

In [1]:
import numpy as np
import pandas as pd
from pathlib import Path

pd.set_option("display.max_columns", 200)

DATA_DIR = Path("../case_study/Data")
PROCESSED_DIR = Path("processed")
PROCESSED_DIR.mkdir(exist_ok=True)

## 1.1 — Load Raw Data

In [2]:
cust    = pd.read_csv(DATA_DIR / "cust_details.csv")
booster = pd.read_csv(DATA_DIR / "SuperBooster.csv")
cxl     = pd.read_csv(DATA_DIR / "cancellations.csv")

print(f"cust_details : {cust.shape}")
print(f"SuperBooster : {booster.shape}")
print(f"cancellations: {cxl.shape}")
cust.head()

cust_details : (125000, 9)
SuperBooster : (25000, 1)
cancellations: (17657, 1)


,customer_id,zip,commune,gender,dob,tenure,internet_usage,tv_product,mobile_product
0,171647760,7,urban,m,1967-03-17,36,Medium,NaN,Low
1,171651484,8,urban,m,1979-08-27,9,Low,Medium,NaN
2,171632004,4,rural,m,1939-05-20,51,Low,NaN,NaN
3,171574581,7,rural,m,1937-02-01,46,High,NaN,Low
4,171586949,8,rural,m,1944-11-16,9,Low,Medium,NaN


## 1.2 — Create Treatment & Outcome Flags via Left Joins

In [3]:
booster_flag = booster.assign(has_booster=1)[["customer_id", "has_booster"]].drop_duplicates()
churn_flag   = cxl.assign(churned=1)[["customer_id", "churned"]].drop_duplicates()

df = cust.merge(booster_flag, on="customer_id", how="left")
df = df.merge(churn_flag,    on="customer_id", how="left")

df["has_booster"] = df["has_booster"].fillna(0).astype(int)
df["churned"]     = df["churned"].fillna(0).astype(int)

print(f"Treatment rate : {df['has_booster'].mean():.2%}")
print(f"Overall churn  : {df['churned'].mean():.2%}")

Treatment rate : 20.00%
Overall churn  : 14.13%


## 1.3 — Feature Engineering

In [4]:
# Age from date of birth — use 2024 as reference year per analytical convention
SNAPSHOT_DATE = pd.Timestamp("2024-01-01")
df["dob"] = pd.to_datetime(df["dob"], errors="coerce")
df["age"] = ((SNAPSHOT_DATE - df["dob"]).dt.days / 365.25).apply(int)

# Age group bins (decade-style for non-linear churn shape check)
df["age_group"] = pd.cut(
    df["age"],
    bins=[0, 25, 35, 50, 65, 120],
    labels=["18-25", "26-35", "36-50", "51-65", "66+"],
    right=True,
)

# Swiss region from ZIP first digit
REGION_MAP = {
    1: "Lake Geneva",
    2: "Bern / Mittelland",
    3: "Bern / Mittelland",
    4: "Northwestern CH",
    5: "Northwestern CH",
    6: "Central Switzerland",
    7: "Graubünden",
    8: "Zürich",
    9: "Eastern Switzerland",
}
df["region"] = df["zip"].astype(int).map(REGION_MAP)

# Fill optional product columns
# Note: 'None' is in pandas default na_values for CSV; use 'no_product' to survive CSV round-trips
df["tv_product"]     = df["tv_product"].fillna("no_product")
df["mobile_product"] = df["mobile_product"].fillna("no_product")

# Standardize free-text fields
for col in ["internet_usage", "commune", "gender", "tv_product", "mobile_product"]:
    df[col] = df[col].astype(str).str.strip()

# Ordered categorical for internet usage
df["internet_usage"] = pd.Categorical(
    df["internet_usage"],
    categories=["Low", "Medium", "High", "Extreme"],
    ordered=True,
)

# Ordered categorical for age_group
df["age_group"] = pd.Categorical(
    df["age_group"].astype(str),
    categories=["18-25", "26-35", "36-50", "51-65", "66+"],
    ordered=True,
)

# Unordered categoricals
for col in ["commune", "gender", "tv_product", "mobile_product", "zip", "region"]:
    df[col] = df[col].astype("category")

## 1.4 — Quality Checks

In [5]:
print("=== Shape ===")
print(df.shape)

print("\n=== Duplicates ===")
print(f"Duplicate customer_ids: {df['customer_id'].duplicated().sum()}")

print("\n=== Missing values ===")
print(df.isnull().sum()[df.isnull().sum() > 0])

print("\n=== Class balance ===")
print(df[["has_booster", "churned"]].apply(pd.Series.value_counts))

df.dtypes

=== Shape ===
(125000, 14)

=== Duplicates ===
Duplicate customer_ids: 0

=== Missing values ===
Series([], dtype: int64)

=== Class balance ===
   has_booster  churned
0       100000   107343
1        25000    17657


customer_id                int64
zip                     category
commune                 category
gender                  category
dob               datetime64[us]
tenure                     int64
internet_usage          category
tv_product              category
mobile_product          category
has_booster                int64
churned                    int64
age                        int64
age_group               category
region                  category
dtype: object

## 1.5 — Save Analysis Table

In [6]:
out_path = PROCESSED_DIR / "df_analysis.csv"
df.to_csv(out_path, index=False)
print(f"Saved {len(df):,} rows → {out_path}")

Saved 125,000 rows → processed/df_analysis.csv
